In [1]:
import re
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.oxml.ns import qn
from pptx.oxml import parse_xml

# --- CONFIGURACIÓN ---
txt_filename = "generate.txt"
ppt_filename = "presentacion_profesional.pptx"

BG_COLOR = RGBColor(209, 209, 209)     # Gris fondo
PURPLE_COLOR = RGBColor(53, 40, 132)   # Azulito/Morado
RED_COLOR = RGBColor(192, 0, 0)        # Rojo

def activar_numeracion_automatica(paragraph, tipo="1"):
    pPr = paragraph._p.get_or_add_pPr()
    buType = "alphaUcPct" if tipo.isalpha() else "arabicPeriod"
    num_xml = f'<a:buAutoNum xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" type="{buType}"/>'
    pPr.insert(0, parse_xml(num_xml))

def aplicar_estilo_run(run, size, color=None, italic=False):
    run.font.name = 'Aptos'
    run.font.bold = True
    run.font.size = Pt(size)
    run.font.italic = italic
    if color:
        run.font.color.rgb = color
    rPr = run._r.get_or_add_rPr()
    rPr.set('spc', '-150') 

def procesar_texto_con_formato(paragraph, texto_sucio):
    """Detecta alineación y aplica colores según el contenedor"""
    
    # 1. Flag para saber si todo el bloque debe ser azulito por las barras ||
    es_bloque_especial = '||' in texto_sucio
    
    # 2. Detectar LISTAS
    match_lista = re.match(r'^([A-Z\d]+)[\.\)]\s+(.*)', texto_sucio)
    
    if match_lista:
        prefijo = match_lista.group(1)
        contenido = match_lista.group(2)
        paragraph.level = 0
        activar_numeracion_automatica(paragraph, tipo=prefijo)
        texto_a_procesar = contenido
        paragraph.alignment = PP_ALIGN.JUSTIFY 
    else:
        # 3. DETECTAR CENTRADO
        if ('{{' in texto_sucio and '}}' in texto_sucio) or ('||' in texto_sucio and '||' in texto_sucio):
            paragraph.alignment = PP_ALIGN.CENTER
        else:
            paragraph.alignment = PP_ALIGN.JUSTIFY
        
        # Limpiamos las barras para procesar el interior
        texto_a_procesar = texto_sucio.replace('||', '')

    # 4. Separar tokens para COLORES
    tokens = re.split(r'(\[\[.*?\]\]|".*?"|\{\{.*?\}\})', texto_a_procesar)
    
    for token in tokens:
        if not token: continue
        run = paragraph.add_run()
        
        # Caso ROJO (Tiene prioridad total)
        if token.startswith('[[') and token.endswith(']]'):
            run.text = token.replace('[[', '').replace(']]', '')
            aplicar_estilo_run(run, size=44, color=RED_COLOR, italic=True)
            
        # Caso AZULITO por comillas
        elif token.startswith('"') and token.endswith('"'):
            run.text = token
            aplicar_estilo_run(run, size=48, color=PURPLE_COLOR, italic=True)
            
        # Caso AZULITO por llaves
        elif token.startswith('{{') and token.endswith('}}'):
            run.text = token.replace('{{', '').replace('}}', '')
            aplicar_estilo_run(run, size=48, color=PURPLE_COLOR, italic=False)
            
        # CASO DE TEXTO RESTANTE
        else:
            run.text = token
            # Si venía de un bloque || ||, lo pintamos azulito. Si no, negro (None).
            color_final = PURPLE_COLOR if es_bloque_especial else None
            aplicar_estilo_run(run, size=48, color=color_final)

def crear_presentacion():
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    try:
        with open(txt_filename, 'r', encoding='utf-8') as f:
            content = f.read()
            slides_data = [s.strip() for s in content.split('\n\n') if s.strip()]
    except FileNotFoundError:
        print("Error: No se encontró el archivo.")
        return

    for block in slides_data:
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        slide.background.fill.solid()
        slide.background.fill.fore_color.rgb = BG_COLOR
        
        left = (prs.slide_width - Inches(13)) / 2
        txBox = slide.shapes.add_textbox(left, Inches(0.5), Inches(13), Inches(6.5))
        tf = txBox.text_frame
        tf.word_wrap = True
        tf.vertical_anchor = MSO_ANCHOR.MIDDLE

        lineas = block.split('\n')
        for i, linea in enumerate(lineas):
            p = tf.paragraphs[i] if i == 0 else tf.add_paragraph()
            procesar_texto_con_formato(p, linea.strip())

    prs.save(ppt_filename)
    print(f"¡Listo! Ahora || || centra y pone todo azulito excepto los [[rojos]].")

if __name__ == "__main__":
    crear_presentacion()

¡Listo! Ahora || || centra y pone todo azulito excepto los [[rojos]].


In [4]:
import re
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.oxml.ns import qn
from pptx.oxml import parse_xml

# --- CONFIGURACIÓN Y CONSTANTES ---
txt_filename = "generate.txt"
ppt_filename = "presentacion_profesional.pptx"

BG_COLOR = RGBColor(209, 209, 209)     # Gris fondo
PURPLE_COLOR = RGBColor(53, 40, 132)   # Azulito/Morado
RED_COLOR = RGBColor(192, 0, 0)        # Rojo

# Pre-compilar Regex para mejor rendimiento
REGEX_LISTA = re.compile(r'^([A-Z\d]+)[\.\)]\s+(.*)')
# Ahora solo buscamos separar por el token rojo
REGEX_TOKENS = re.compile(r'(\[\[.*?\]\])') 

def activar_numeracion_automatica(paragraph, tipo="1"):
    pPr = paragraph._p.get_or_add_pPr()
    buType = "alphaUcPct" if tipo.isalpha() else "arabicPeriod"
    num_xml = f'<a:buAutoNum xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" type="{buType}"/>'
    pPr.insert(0, parse_xml(num_xml))

def aplicar_estilo_run(run, size, color=None, italic=False):
    run.font.name = 'Aptos'
    run.font.bold = True
    run.font.size = Pt(size)
    run.font.italic = italic
    if color:
        run.font.color.rgb = color
    rPr = run._r.get_or_add_rPr()
    rPr.set('spc', '-150') 

def procesar_texto_con_formato(paragraph, texto_sucio):
    """Detecta alineación y aplica colores según el contenedor"""
    
    # 1. Detectar si es un bloque especial centrado/morado
    es_bloque_especial = '||' in texto_sucio
    
    # 2. Detectar LISTAS
    match_lista = REGEX_LISTA.match(texto_sucio)
    
    if match_lista:
        prefijo, contenido = match_lista.groups()
        paragraph.level = 0
        activar_numeracion_automatica(paragraph, tipo=prefijo)
        texto_a_procesar = contenido
        paragraph.alignment = PP_ALIGN.JUSTIFY 
    else:
        # 3. Alineación: Si tiene || va centrado, si no, justificado
        paragraph.alignment = PP_ALIGN.CENTER if es_bloque_especial else PP_ALIGN.JUSTIFY
        # Limpiamos las barras para procesar el interior
        texto_a_procesar = texto_sucio.replace('||', '')

    # 4. Separar tokens. Ahora solo nos preocupamos por aislar los [[rojos]]
    tokens = REGEX_TOKENS.split(texto_a_procesar)
    
    for token in tokens:
        if not token: continue
        
        run = paragraph.add_run()
        
        # Caso ROJO (Tiene prioridad total)
        if token.startswith('[[') and token.endswith(']]'):
            run.text = token.replace('[[', '').replace(']]', '')
            aplicar_estilo_run(run, size=44, color=RED_COLOR, italic=True)
            
        # CASO DE TEXTO RESTANTE (Normal o Morado si venía de || ||)
        else:
            run.text = token
            color_final = PURPLE_COLOR if es_bloque_especial else None
            aplicar_estilo_run(run, size=48, color=color_final)

def crear_presentacion():
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    try:
        with open(txt_filename, 'r', encoding='utf-8') as f:
            content = f.read()
            slides_data = [s.strip() for s in content.split('\n\n') if s.strip()]
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{txt_filename}'.")
        return

    for block in slides_data:
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        slide.background.fill.solid()
        slide.background.fill.fore_color.rgb = BG_COLOR
        
        left = (prs.slide_width - Inches(13)) / 2
        txBox = slide.shapes.add_textbox(left, Inches(0.5), Inches(13), Inches(6.5))
        tf = txBox.text_frame
        tf.word_wrap = True
        tf.vertical_anchor = MSO_ANCHOR.MIDDLE

        lineas = block.split('\n')
        for i, linea in enumerate(lineas):
            p = tf.paragraphs[i] if i == 0 else tf.add_paragraph()
            procesar_texto_con_formato(p, linea.strip())

    prs.save(ppt_filename)
    print(f"¡Listo! Se ha simplificado la sintaxis. Ahora || || centra y pone todo azulito excepto los [[rojos]]. Las comillas funcionan normal.")

if __name__ == "__main__":
    crear_presentacion()

¡Listo! Se ha simplificado la sintaxis. Ahora || || centra y pone todo azulito excepto los [[rojos]]. Las comillas funcionan normal.
